<a href="https://colab.research.google.com/github/12leocheung/blocks/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, os
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.environ.setdefault("TORCH_COMPILE_DISABLE", "1")


# ================================================================
# FILE SEARCH HELPER
# ================================================================
def find_file(name):
    """
    If `name` is already a valid path, return it as-is.
    Otherwise, search for a file with that name starting from the
    current working directory and walking up through every parent
    directory until the filesystem root. Returns the first match
    found, or the original name if nothing is found (so existing
    error handling still fires).
    """
    if os.path.exists(name):
        return name
    search = os.path.basename(name)
    current = os.path.abspath(os.getcwd())
    visited = set()
    while True:
        if current in visited:
            break
        visited.add(current)
        candidate = os.path.join(current, search)
        if os.path.exists(candidate):
            print(f"[INFO] Found '{search}' at: {candidate}")
            return candidate
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    print(f"[WARN] Could not find '{search}' in any parent directory.")
    return name


import torch, time, math, hashlib
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torch.amp import autocast
from tokenizers import Tokenizer, models, trainers
import traceback

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

torch.set_float32_matmul_precision("high")

# ================================================================
# DEFAULT CONFIG
# ================================================================
BLOCK_SIZE  = 256
DIM         = 384
LAYERS      = 8
HEADS       = 6
KV_HEADS    = 6
FF_HIDDEN   = 1344
VOCAB_SIZE  = 32000
BATCH_SIZE  = 32
ACCUM_STEPS = 2

PRESET_CONFIGS = {
    "small": dict(BLOCK_SIZE=256,  DIM=384,  LAYERS=8,  HEADS=6,  KV_HEADS=6,
                  FF_HIDDEN=1344,  VOCAB_SIZE=32000, BATCH_SIZE=32, ACCUM_STEPS=2),
    "8b":    dict(BLOCK_SIZE=2048, DIM=4096, LAYERS=32, HEADS=32, KV_HEADS=8,
                  FF_HIDDEN=14336, VOCAB_SIZE=32000, BATCH_SIZE=4,  ACCUM_STEPS=8),
}

def apply_preset(name):
    g = globals()
    for k, v in PRESET_CONFIGS[name].items():
        g[k] = v

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")


# ================================================================
# TOKENIZER
# ================================================================
def build_tokenizer(path):
    if os.path.exists("tokenizer.json"):
        print("[INFO] Loading existing tokenizer.json")
        return Tokenizer.from_file("tokenizer.json")
    print("[INFO] Training tokenizer ...")
    tok = Tokenizer(models.BPE())
    from tokenizers.pre_tokenizers import ByteLevel
    from tokenizers.decoders    import ByteLevel as BLD
    tok.pre_tokenizer = ByteLevel()
    tok.decoder       = BLD()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=["<|endoftext|>", "<|pad|>"]
    )
    tok.train([path], trainer)
    tok.save("tokenizer.json")
    print("[INFO] Tokenizer saved.")
    return tok


# ================================================================
# DATA
# ================================================================
def _file_hash(path, n=65536):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read(n))
    return h.hexdigest()[:8]

def parse_convo_data(text):
    import re
    pairs, lines, i = [], text.splitlines(), 0
    while i < len(lines):
        q = re.match(r'^- - (.+)', lines[i].rstrip())
        if q:
            question = q.group(1).strip(); i += 1
            while i < len(lines):
                a = re.match(r'^  - (.+)', lines[i].rstrip())
                if a:
                    answer = a.group(1).strip()
                    if question and answer:
                        pairs.append(f"User: {question}\nAI: {answer}\n<|endoftext|>")
                    i += 1
                else:
                    break
        else:
            i += 1
    if len(pairs) < 10:
        print(f"[INFO] Only {len(pairs)} Q&A pairs found — using raw text")
        return text
    result = "\n".join(pairs)
    print(f"[INFO] Extracted {len(pairs)} Q&A pairs -> {len(result):,} chars")
    return result


class PackedDataset(Dataset):
    """
    Packs all token IDs into one continuous stream then slices into
    block_size chunks — zero padding waste, more signal per step.
    Cache filename includes a hash of the data file so stale caches
    are never silently reused.
    """
    def __init__(self, text, tokenizer, block_size, data_hash=""):
        cache = f"packed_b{block_size}_{data_hash}.pt"
        if os.path.exists(cache):
            print(f"[INFO] Loading cached samples from {cache}")
            self.samples = torch.load(cache, weights_only=True)
        else:
            print("[INFO] Packing token stream ...")
            eot_id = tokenizer.token_to_id("<|endoftext|>") or 0
            exchanges = [e.strip() for e in text.split("<|endoftext|>") if e.strip()]
            stream = []
            for ex in exchanges:
                stream += tokenizer.encode(ex).ids + [eot_id]
            if len(stream) < block_size + 1:
                raise ValueError(f"Data too small: only {len(stream)} tokens total.")
            samples = []
            for i in range(0, len(stream) - block_size, block_size):
                samples.append(stream[i : i + block_size + 1])
            self.samples = torch.tensor(samples, dtype=torch.long)
            torch.save(self.samples, cache)
            print(f"[INFO] Packed {len(self.samples)} samples -> {cache}")
        print(f"[INFO] Dataset: {len(self)} samples, block={block_size}")

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i):
        r = self.samples[i]; return r[:-1], r[1:]


# ================================================================
# ROPE
# ================================================================
def precompute_rope(head_dim, seq_len, base=500_000):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t     = torch.arange(seq_len).float()
    freqs = torch.outer(t, theta)
    return (torch.cos(freqs).unsqueeze(0).unsqueeze(0),
            torch.sin(freqs).unsqueeze(0).unsqueeze(0))

def apply_rope(x, cos, sin):
    T = x.shape[2]
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack([x1*cos[:,:,:T] - x2*sin[:,:,:T],
                        x1*sin[:,:,:T] + x2*cos[:,:,:T]], dim=-1).flatten(-2)


# ================================================================
# TRANSFORMER BLOCK  (GQA + SwiGLU)
# ================================================================
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, kv_heads, ff_hidden, block_size):
        super().__init__()
        assert heads % kv_heads == 0
        self.heads    = heads
        self.kv_heads = kv_heads
        self.head_dim = dim // heads
        self.repeat   = heads // kv_heads

        self.wq = nn.Linear(dim, heads    * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)

        self.ff_gate = nn.Linear(dim,       ff_hidden * 2, bias=False)
        self.ff_down = nn.Linear(ff_hidden, dim,           bias=False)

        self.ln1  = nn.RMSNorm(dim, eps=1e-5)
        self.ln2  = nn.RMSNorm(dim, eps=1e-5)
        self.drop = nn.Dropout(0.05)

        cos, sin = precompute_rope(self.head_dim, block_size * 2)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, kv_cache=None):
        B, T, C = x.shape
        h = self.ln1(x)
        q = self.wq(h).view(B, T, self.heads,    self.head_dim).transpose(1, 2)
        k = self.wk(h).view(B, T, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(h).view(B, T, self.kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        # KV cache support
        new_cache = None
        if kv_cache is not None:
            pk, pv   = kv_cache
            k        = torch.cat([pk, k], dim=2)
            v        = torch.cat([pv, v], dim=2)
            new_cache = (k, v)

        if self.repeat > 1:
            k = k.repeat_interleave(self.repeat, dim=1)
            v = v.repeat_interleave(self.repeat, dim=1)

        is_causal = (kv_cache is None)   # incremental decode is already causal
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=is_causal,
            dropout_p=0.05 if self.training else 0.0
        )
        x = x + self.wo(out.transpose(1, 2).reshape(B, T, C))
        g1, g2 = self.ff_gate(self.ln2(x)).chunk(2, dim=-1)
        x = x + self.drop(self.ff_down(F.silu(g1) * g2))
        return x, new_cache


# ================================================================
# MODEL
# ================================================================
class BetterGPT(nn.Module):
    def __init__(self, vocab, dim, layers, heads, kv_heads, ff_hidden, block_size):
        super().__init__()
        self.embed  = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads, kv_heads, ff_hidden, block_size)
            for _ in range(layers)
        ])
        self.ln   = nn.RMSNorm(dim, eps=1e-5)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.embed.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, x, kv_caches=None):
        x       = self.embed(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            cache = kv_caches[i] if kv_caches else None
            x, nc = blk(x, cache)
            new_caches.append(nc)
        return self.head(self.ln(x)), new_caches


def make_model(vocab, dim, layers, heads, kv_heads, ff_hidden, block_size):
    return BetterGPT(vocab=vocab, dim=dim, layers=layers, heads=heads,
                     kv_heads=kv_heads, ff_hidden=ff_hidden, block_size=block_size)


# ================================================================
# CHECKPOINT
# ================================================================
def save_checkpoint(model, tokenizer, path, epoch=None, opt_state=None,
                    val_loss=None, is_best=False):
    tok_json = tokenizer.to_str() if hasattr(tokenizer, "to_str") else None
    raw  = getattr(model, "_orig_mod", model)
    b0   = raw.blocks[0]
    payload = {
        "model_state":    {k: v.cpu() for k, v in raw.state_dict().items()},
        "vocab_size":     raw.embed.weight.shape[0],
        "dim":            raw.embed.weight.shape[1],
        "layers":         len(raw.blocks),
        "heads":          b0.heads,
        "kv_heads":       b0.kv_heads,
        "ff_hidden":      b0.ff_down.weight.shape[1],
        "block_size":     b0.rope_cos.shape[2],
        "tokenizer_json": tok_json,
    }
    if epoch     is not None: payload["epoch"]     = epoch
    if opt_state is not None: payload["opt_state"] = opt_state
    if val_loss  is not None: payload["val_loss"]  = val_loss
    torch.save(payload, path)
    tag = " [BEST]" if is_best else ""
    vl  = f"  val={val_loss:.4f}" if val_loss is not None else ""
    print(f"[INFO] Saved{tag} -> {path}{vl}")


def load_checkpoint(path):
    import re
    data = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(data, dict) and "model_state" in data:
        raw = data["model_state"]
    elif isinstance(data, dict) and "state_dict" in data:
        raw = data["state_dict"]
    else:
        raw = data

    KEY_MAP = {
        "token_emb.weight":      "embed.weight",
        "tok_embeddings.weight": "embed.weight",
        "ln_f.weight": "ln.weight", "ln_f.bias": "ln.bias",
        "norm.weight": "ln.weight", "norm.bias":  "ln.bias",
        "output.weight": "head.weight", "lm_head.weight": "head.weight",
    }
    state = {}
    for k, v in raw.items():
        k2 = KEY_MAP.get(k, k)
        k2 = re.sub(r"^_orig_mod\.", "", k2)
        k2 = re.sub(r"^(blocks\.\d+)\.attention_norm\.", r"\1.ln1.", k2)
        k2 = re.sub(r"^(blocks\.\d+)\.ffn_norm\.",       r"\1.ln2.", k2)
        k2 = re.sub(r"^(blocks\.\d+)\.norm1\.",           r"\1.ln1.", k2)
        k2 = re.sub(r"^(blocks\.\d+)\.norm2\.",           r"\1.ln2.", k2)
        state[k2] = v

    def _get(key, fallback):
        return data[key] if isinstance(data, dict) and key in data else fallback

    ew       = state.get("embed.weight")
    vocab    = _get("vocab_size",  int(ew.shape[0]) if ew is not None else VOCAB_SIZE)
    dim      = _get("dim",         int(ew.shape[1]) if ew is not None else DIM)
    indices  = sorted({int(k.split(".")[1]) for k in state if k.startswith("blocks.")})
    layers   = _get("layers",  (max(indices)+1) if indices else LAYERS)
    heads    = _get("heads",   max(1, dim // 64))
    kv_heads = _get("kv_heads", heads)

    ff_key    = next((k for k in state if k.endswith(".ff_down.weight")), None)
    ff_hidden = _get("ff_hidden",
                     int(state[ff_key].shape[1]) if ff_key else FF_HIDDEN)

    rope_key   = next((k for k in state if k.endswith(".rope_cos")), None)
    block_size = _get("block_size",
                      int(state[rope_key].shape[2]) if rope_key else BLOCK_SIZE)

    print(f"[INFO] Arch  : vocab={vocab}, dim={dim}, layers={layers}, "
          f"heads={heads}, kv={kv_heads}, ff={ff_hidden}, block={block_size}")

    model = make_model(vocab=vocab, dim=dim, layers=layers, heads=heads,
                       kv_heads=kv_heads, ff_hidden=ff_hidden, block_size=block_size)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:    print(f"[WARN] Missing     ({len(missing)}): {missing[:4]}")
    if unexpected: print(f"[WARN] Unexpected  ({len(unexpected)}): {unexpected[:4]}")

    tokenizer = None
    if _get("tokenizer_json", None):
        try:
            tokenizer = Tokenizer.from_str(data["tokenizer_json"])
            print("[INFO] Tokenizer loaded from checkpoint")
        except Exception as e:
            print(f"[WARN] Tokenizer not loaded: {e}")

    opt_state   = _get("opt_state",  None)
    start_epoch = _get("epoch",      0)
    best_val    = _get("val_loss",   float("inf"))
    return model, tokenizer, opt_state, start_epoch, block_size, best_val


# ================================================================
# HELPERS
# ================================================================
def fmt_time(s):
    s = int(s)
    if s < 60:   return f"{s}s"
    if s < 3600: return f"{s//60}m {s%60}s"
    return f"{s//3600}h {(s%3600)//60}m"

def _nw():
    return 0 if DEVICE.type != "cpu" else min(2, os.cpu_count() or 1)

def _make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=_nw(), pin_memory=(DEVICE.type == "cuda"),
                      persistent_workers=(_nw() > 0))

def _cosine_lr(opt, total_steps, warmup_steps, min_ratio=0.1):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)


# ================================================================
# LOAD  (fresh training)
# ================================================================
def load_data(path, num_threads=4):
    if DEVICE.type == "cpu": torch.set_num_threads(num_threads)
    print(f"[INFO] Device : {str(DEVICE).upper()}")
    print(f"[INFO] Reading: {path}")
    with open(path, encoding="utf-8") as f:
        text = f.read(2_000_000)
    print(f"[INFO] Read {len(text):,} chars")
    text      = parse_convo_data(text)
    fhash     = _file_hash(path)
    tokenizer = build_tokenizer(path)
    dataset   = PackedDataset(text, tokenizer, block_size=BLOCK_SIZE, data_hash=fhash)
    if len(dataset) < 10:
        raise ValueError(f"Dataset too small ({len(dataset)} samples).")
    model = make_model(vocab=tokenizer.get_vocab_size(), dim=DIM, layers=LAYERS,
                       heads=HEADS, kv_heads=KV_HEADS, ff_hidden=FF_HIDDEN,
                       block_size=BLOCK_SIZE).to(DEVICE)
    if DEVICE.type == "cuda":
        try:   model = torch.compile(model, backend="inductor"); print("[INFO] torch.compile OK")
        except Exception as e: print(f"[INFO] torch.compile skipped: {e}")
    params = sum(p.numel() for p in model.parameters())
    print(f"[INFO] Params : {params/1e6:.1f}M  ({params:,})")
    return tokenizer, dataset, model


# ================================================================
# RESUME
# ================================================================
def load_resume(ckpt_path, data_path=None, num_threads=4):
    if DEVICE.type == "cpu": torch.set_num_threads(num_threads)
    model, tokenizer, opt_state, start_epoch, block_size, best_val = \
        load_checkpoint(ckpt_path)

    if data_path and os.path.exists(data_path):
        dp = data_path
    else:
        ckpt_dir = os.path.dirname(os.path.abspath(ckpt_path))
        dp = next((p for p in [os.path.join(ckpt_dir, "data.txt"),
                                os.path.join(os.getcwd(), "data.txt")]
                   if os.path.exists(p)), None)
    if dp is None:
        raise FileNotFoundError("No data.txt found beside checkpoint or in cwd.")
    print(f"[INFO] Data   : {dp}")
    with open(dp, encoding="utf-8") as f:
        text = f.read(2_000_000)
    text = parse_convo_data(text)
    if tokenizer is None:
        tokenizer = build_tokenizer(dp)

    fhash   = _file_hash(dp)
    dataset = PackedDataset(text, tokenizer, block_size=block_size, data_hash=fhash)
    if len(dataset) < 10:
        raise ValueError(f"Dataset too small ({len(dataset)} samples).")
    model = model.to(DEVICE)
    if DEVICE.type == "cuda":
        try:   model = torch.compile(model, backend="inductor")
        except Exception: pass

    return tokenizer, dataset, model, opt_state, start_epoch, best_val


# ================================================================
# TRAIN
# ================================================================
def train(model, dataset, tokenizer, epochs, lr, num_threads=4,
          opt_state=None, start_epoch=0, save_path="save.pt",
          patience=12, prior_best_val=float("inf")):

    if DEVICE.type == "cpu": torch.set_num_threads(num_threads)
    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable: raise RuntimeError("No trainable parameters!")
    print(f"[INFO] Trainable: {sum(p.numel() for p in trainable)/1e6:.1f}M")
    print(f"[INFO] Epochs {start_epoch+1}-{start_epoch+epochs}  |  "
          f"LR {lr}  |  Patience {patience}  |  Device {DEVICE}")

    opt = torch.optim.AdamW(trainable, lr=lr,
                            betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8)
    if opt_state:
        try:
            opt.load_state_dict(opt_state)
            print("[INFO] Optimizer state restored")
        except Exception as e:
            print(f"[WARN] Could not restore optimizer: {e}")

    n = len(dataset)
    val_n = max(1, n - max(1, int(0.9 * n)))
    train_ds, val_ds = random_split(dataset, [n - val_n, val_n])
    train_loader = _make_loader(train_ds, shuffle=True)
    val_loader   = _make_loader(val_ds,   shuffle=False)

    total_batches = len(train_loader)
    total_steps   = epochs * max(1, total_batches)
    warmup_steps  = max(1, int(0.05 * total_steps))
    scheduler     = _cosine_lr(opt, total_steps, warmup_steps)

    pad_id       = (tokenizer.token_to_id("<|pad|>") or 0) if tokenizer else 0
    use_cuda_amp = DEVICE.type == "cuda"
    use_bf16_cpu = DEVICE.type == "cpu"
    scaler       = torch.amp.GradScaler("cuda", enabled=use_cuda_amp)

    def amp_ctx():
        if use_cuda_amp:   return autocast("cuda", dtype=torch.bfloat16)
        elif use_bf16_cpu: return autocast("cpu",  dtype=torch.bfloat16)
        from contextlib import nullcontext; return nullcontext()

    best_val    = prior_best_val
    best_epoch  = start_epoch
    bad_epochs  = 0
    losses      = []
    t0          = time.time()

    # derive best checkpoint path  (e.g. model_best.pt alongside model.pt)
    base, ext    = os.path.splitext(save_path)
    best_path    = f"{base}_best{ext}"

    print(f"[INFO] Best checkpoint -> {best_path}")
    print(f"[INFO] Batches/epoch {total_batches}  |  Accum {ACCUM_STEPS}  |  "
          f"Warmup {warmup_steps} steps")
    print(f"[INFO] Ctrl+C to stop early.\n")
    print(f"  {'Epoch':>6}  {'Val Loss':>9}  {'PPL':>7}  {'Delta':>8}  "
          f"{'LR':>9}  {'Ep':>7}  {'Total':>8}  {'':>6}")
    print("  " + "─" * 72)

    try:
        for ep in range(epochs):
            model.train(); opt.zero_grad(set_to_none=True)
            ep_start = time.time()

            it = tqdm(enumerate(train_loader), total=total_batches,
                      desc=f"  Ep {start_epoch+ep+1:>3}", ncols=84,
                      unit="batch", leave=False) if HAS_TQDM \
                 else enumerate(train_loader)

            for bi, (x, y) in it:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                with amp_ctx():
                    logits, _ = model(x)
                    loss = F.cross_entropy(
                        logits.view(-1, logits.size(-1)),
                        y.view(-1), ignore_index=pad_id
                    ) / ACCUM_STEPS
                if use_cuda_amp: scaler.scale(loss).backward()
                else:            loss.backward()

                if (bi + 1) % ACCUM_STEPS == 0 or (bi + 1) == total_batches:
                    if use_cuda_amp:
                        scaler.unscale_(opt)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(opt); scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step()
                    scheduler.step(); opt.zero_grad(set_to_none=True)

                if not HAS_TQDM:
                    pct = (bi+1)/total_batches*100
                    print(f"\r  Ep {start_epoch+ep+1}  {pct:5.1f}%"
                          f" ({bi+1}/{total_batches})", end="", flush=True)
            if not HAS_TQDM: print()

            # ── validation ──────────────────────────────────
            model.eval(); val_loss = 0.0
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(DEVICE, non_blocking=True)
                    y = y.to(DEVICE, non_blocking=True)
                    with amp_ctx():
                        out, _ = model(x)
                        val_loss += F.cross_entropy(
                            out.view(-1, out.size(-1)),
                            y.view(-1), ignore_index=pad_id
                        ).item()
            val_loss /= len(val_loader)
            ppl   = math.exp(min(val_loss, 20))
            losses.append(val_loss)
            delta = f"{val_loss-losses[-2]:+.4f}" if len(losses) > 1 else "      --"
            cur_lr = opt.param_groups[0]["lr"]

            # ── early stopping + best-model save ────────────
            improved = val_loss < (best_val - 1e-4)
            if improved:
                best_val   = val_loss
                best_epoch = start_epoch + ep + 1
                bad_epochs = 0
                save_checkpoint(model, tokenizer, best_path,
                                epoch=best_epoch,
                                opt_state=opt.state_dict(),
                                val_loss=best_val,
                                is_best=True)
                flag = " <-- best"
            else:
                bad_epochs += 1
                flag = f" (no improvement {bad_epochs}/{patience})"

            print(f"  {start_epoch+ep+1:>6}  {val_loss:>9.4f}  {ppl:>7.1f}  "
                  f"{delta:>8}  {cur_lr:>9.6f}  {fmt_time(time.time()-ep_start):>7}  "
                  f"{fmt_time(time.time()-t0):>8}{flag}")

            if bad_epochs >= patience:
                print(f"\n[INFO] Early stopping: no improvement for {patience} epochs.")
                print(f"[INFO] Best was epoch {best_epoch}  val={best_val:.4f}  "
                      f"ppl={math.exp(min(best_val,20)):.1f}")
                break

    except KeyboardInterrupt:
        print("\n\n[INFO] Stopped early by user.")

    print("  " + "─" * 72)
    print(f"[INFO] Best val loss: {best_val:.4f}  (epoch {best_epoch})  "
          f"| PPL: {math.exp(min(best_val,20)):.1f}  "
          f"| Total: {fmt_time(time.time()-t0)}")
    print(f"[INFO] Best model saved at: {best_path}\n")
    return model, best_path


# ================================================================
# GENERATE  (with KV cache)
# ================================================================
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new=150, temp=0.4,
             top_k=30, top_p=0.85, rep_penalty=1.15, history=None):
    model.eval()
    raw = getattr(model, "_orig_mod", model)
    buf = getattr(raw.blocks[0], "rope_cos", None)
    bsz = buf.shape[2] if buf is not None else BLOCK_SIZE

    # build prompt exactly matching training template
    ctx = []
    if history:
        for role, txt in history[-6:]:
            ctx.append(f"{'User' if role=='user' else 'AI'}: {txt}")
    ctx.append(f"User: {prompt}\nAI:")
    prompt_ids = tokenizer.encode("\n".join(ctx)).ids[-bsz:]

    x         = torch.tensor(prompt_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    eot_ids   = set(tokenizer.encode("<|endoftext|>").ids)
    nl_ids    = set(tokenizer.encode("\n").ids)
    recent    = list(prompt_ids[-64:])   # sliding window for rep penalty
    gen       = []
    kv_caches = None

    # prefill
    _, kv_caches = model(x, kv_caches=None)

    # incremental decode
    next_tok = x[:, -1:]
    for _ in range(max_new):
        logits, kv_caches = model(next_tok, kv_caches=kv_caches)
        logits = logits[:, -1, :].float()

        # repetition penalty on recent window only
        if rep_penalty != 1.0:
            for t in set(recent):
                if t < logits.size(-1):
                    logits[0, t] = logits[0, t] / rep_penalty \
                                   if logits[0, t] > 0 \
                                   else logits[0, t] * rep_penalty

        logits /= max(temp, 1e-6)

        if top_k:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, -1:]] = -1e9
        if top_p < 1.0:
            sl, si = torch.sort(logits, descending=True)
            cp     = torch.cumsum(F.softmax(sl, -1), -1)
            sl[cp - F.softmax(sl, -1) > top_p] = -1e9
            logits.scatter_(1, si, sl)

        tid = torch.multinomial(F.softmax(logits, -1), 1).item()
        gen.append(tid)
        recent.append(tid)
        if len(recent) > 64: recent.pop(0)
        next_tok = torch.tensor([[tid]], device=DEVICE)

        if tid in eot_ids or tid in nl_ids:
            break

    reply = tokenizer.decode(gen).strip()
    for stop in ("User:", "\nUser", "<|endoftext|>"):
        if stop in reply:
            reply = reply[:reply.index(stop)].strip()
    return reply


# ================================================================
# CHAT REPL
# ================================================================
def chat_repl(model, tokenizer, temp=0.4, top_k=30, top_p=0.85,
              rep_penalty=1.15, max_tokens=150):
    print("\n" + "=" * 54)
    print("    LLM Chat  |  'quit' to exit  |  'clear' to reset")
    print("  Tip: lower temp = more consistent, higher = creative")
    print("=" * 54 + "\n")
    history = []
    while True:
        try:   user = input("You: ").strip()
        except (EOFError, KeyboardInterrupt): print("\n[INFO] Bye!"); break
        if not user: continue
        if user.lower() in ("quit", "exit"): print("[INFO] Bye!"); break
        if user.lower() == "clear":
            history = []; print("[INFO] History cleared.\n"); continue
        if user.lower().startswith("/temp "):
            try:   temp = float(user.split()[1]); print(f"[INFO] temp={temp}")
            except ValueError: print("[WARN] Usage: /temp 0.4")
            continue
        if user.lower().startswith("/topk "):
            try:   top_k = int(user.split()[1]); print(f"[INFO] top_k={top_k}")
            except ValueError: print("[WARN] Usage: /topk 30")
            continue

        history.append(("user", user))
        reply = generate(model, tokenizer, user,
                         max_new=max_tokens, temp=temp,
                         top_k=top_k, top_p=top_p,
                         rep_penalty=rep_penalty,
                         history=history[:-1])
        history.append(("ai", reply or "..."))
        print(f"AI: {reply or '...'}\n")


# ================================================================
# INTERACTIVE MENU
# ================================================================
def interactive_prompt():
    print("\n    LLM")
    print("  " + "=" * 34)
    print("  [1] Train from scratch")
    print("  [2] Resume from checkpoint")
    print("  [3] Chat with saved model")
    choice = input("  Choice [1]: ").strip() or "1"
    cfg = {"mode": {"1":"train","2":"resume","3":"chat"}.get(choice, "train")}

    if cfg["mode"] in ("resume", "chat"):
        cfg["ckpt"] = find_file(input("  Checkpoint .pt path: ").strip())
        if not cfg["ckpt"]: print("[ERROR] Required"); sys.exit(1)
    else:
        cfg["ckpt"] = None

    if cfg["mode"] == "train":
        cfg["data"] = find_file(input("  Path to data.txt: ").strip())
        if not cfg["data"]: print("[ERROR] Required"); sys.exit(1)
    elif cfg["mode"] == "resume":
        cfg["data"] = (lambda p: find_file(p) if p else None)(input("  data.txt path (blank=auto): ").strip())
    else:
        cfg["data"] = None

    if cfg["mode"] in ("train", "resume"):
        cfg["epochs"]   = int(input("  Epochs            [50]: ").strip() or "50")
        cfg["lr"]       = float(input("  LR            [1e-4]: ").strip() or "1e-4")
        cfg["patience"] = int(input("  Early stop patience [12]: ").strip() or "12")
        cfg["threads"]  = int(input("  CPU threads         [4]: ").strip() or "4")
        cfg["save"]     = input("  Save to [save.pt]: ").strip() \
                          or cfg.get("ckpt") or "save.pt"
        cfg["preset"]   = input("  Preset [small/8b, blank=small]: ").strip() or "small"
    else:
        cfg["epochs"] = cfg["lr"] = cfg["patience"] = None
        cfg["threads"] = int(input("  CPU threads [4]: ").strip() or "4")
        cfg["save"] = cfg["preset"] = None

    if cfg["mode"] == "chat":
        print("  (defaults tuned for small models — lower temp = less gibberish)")
        cfg["temp"]        = float(input("  Temperature    [0.4]: ").strip() or "0.4")
        cfg["top_k"]       = int(input("  Top-K           [30]: ").strip() or "30")
        cfg["top_p"]       = float(input("  Top-P         [0.85]: ").strip() or "0.85")
        cfg["rep_penalty"] = float(input("  Rep penalty   [1.15]: ").strip() or "1.15")
        cfg["max_tokens"]  = int(input("  Max tokens     [150]: ").strip() or "150")
    else:
        cfg.setdefault("temp", 0.4); cfg.setdefault("top_k", 30)
        cfg.setdefault("top_p", 0.85); cfg.setdefault("rep_penalty", 1.15)
        cfg.setdefault("max_tokens", 150)

    print()
    return cfg


# ================================================================
# MAIN
# ================================================================
def main():
    _skip = ("--KernelManager","--ip=","--stdin=","--control=","--hb=",
             "--Session","--shell=","--transport=","--iopub=","--heartbeat=","-f")
    _argv = [a for a in sys.argv[1:]
             if not (a.endswith(".json") and "kernel" in a)
             and not any(a.startswith(p) for p in _skip)]

    if _argv:
        import argparse
        p = argparse.ArgumentParser(description="LLM — GPT trainer & chat")
        p.add_argument("--data",        metavar="PATH")
        p.add_argument("--resume",      metavar="PATH")
        p.add_argument("--chat",        metavar="PATH")
        p.add_argument("--epochs",      type=int,   default=50)
        p.add_argument("--lr",          type=float, default=1e-4)
        p.add_argument("--patience",    type=int,   default=12)
        p.add_argument("--threads",     type=int,   default=4)
        p.add_argument("--save",        default="save.pt")
        p.add_argument("--preset",      default="small", choices=["small","8b"])
        p.add_argument("--temp",        type=float, default=0.4)
        p.add_argument("--top-k",       type=int,   default=30,   dest="top_k")
        p.add_argument("--top-p",       type=float, default=0.85, dest="top_p")
        p.add_argument("--rep-penalty", type=float, default=1.15, dest="rep_penalty")
        p.add_argument("--max-tokens",  type=int,   default=150,  dest="max_tokens")
        a = p.parse_args(_argv)
        if not a.data and not a.resume and not a.chat:
            p.error("Provide --data, --resume, or --chat.")
        cfg = dict(
            mode="chat" if a.chat else ("resume" if a.resume else "train"),
            ckpt=find_file(a.chat or a.resume) if (a.chat or a.resume) else None, data=find_file(a.data) if a.data else None,
            epochs=a.epochs, lr=a.lr, patience=a.patience,
            threads=a.threads, save=a.save, preset=a.preset,
            temp=a.temp, top_k=a.top_k, top_p=a.top_p,
            rep_penalty=a.rep_penalty, max_tokens=a.max_tokens,
        )
    else:
        cfg = interactive_prompt()

    if cfg.get("preset") and cfg["mode"] != "chat":
        apply_preset(cfg["preset"])

    print(f"\n  LLM — {str(DEVICE).upper()}")
    print("  " + "=" * 38)

    try:
        if cfg["mode"] == "train":
            tokenizer, dataset, model = load_data(cfg["data"], cfg["threads"])
            train(model, dataset, tokenizer,
                  cfg["epochs"], cfg["lr"], cfg["threads"],
                  save_path=cfg["save"],
                  patience=cfg.get("patience", 12))

        elif cfg["mode"] == "resume":
            tokenizer, dataset, model, opt_state, start_ep, best_val = \
                load_resume(cfg["ckpt"], cfg.get("data"), cfg["threads"])
            train(model, dataset, tokenizer,
                  cfg["epochs"], cfg["lr"], cfg["threads"],
                  opt_state=opt_state, start_epoch=start_ep,
                  save_path=cfg.get("save") or cfg["ckpt"],
                  patience=cfg.get("patience", 12),
                  prior_best_val=best_val)

        elif cfg["mode"] == "chat":
            if DEVICE.type == "cpu":
                torch.set_num_threads(cfg.get("threads", 4))
            model, tokenizer, _, _, _, _ = load_checkpoint(cfg["ckpt"])
            if tokenizer is None:
                if os.path.exists("tokenizer.json"):
                    tokenizer = Tokenizer.from_file("tokenizer.json")
                else:
                    print("[ERROR] No tokenizer found."); sys.exit(1)
            model = model.to(DEVICE)
            chat_repl(model, tokenizer,
                      temp=cfg["temp"], top_k=cfg["top_k"],
                      top_p=cfg["top_p"], rep_penalty=cfg["rep_penalty"],
                      max_tokens=cfg["max_tokens"])

    except FileNotFoundError as e:
        print(f"\n[ERROR] {e}"); sys.exit(1)
    except Exception:
        traceback.print_exc(); sys.exit(1)


if __name__ == "__main__":
    main()



    LLM
  [1] Train from scratch
  [2] Resume from checkpoint
  [3] Chat with saved model
